<a href="https://colab.research.google.com/github/Ezliene/FYP-Solar-Irradience-Prediction/blob/main/solar_isdkh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import numpy as np, pandas as pd
P = '/content/drive/MyDrive/fyp_solar/'

X_train = np.load(P+'train_X_24h_v2.npy')
X_val   = np.load(P+'val_X_24h_v2.npy')
X_test  = np.load(P+'test_X_24h_v2.npy')
y_train = np.load(P+'train_y_24h_v2.npy')
y_val   = np.load(P+'val_y_24h_v2.npy')
y_test  = np.load(P+'test_y_24h_v2.npy')

val_meta  = pd.read_csv(P+'val_meta_24h.xls')
test_meta = pd.read_csv(P+'test_meta_24h.xls')
scaler    = pd.read_csv(P+'solar_scaler_stats.xls')

for n, a in [('X_train',X_train),('X_val',X_val),('X_test',X_test),
             ('y_train',y_train),('y_val',y_val),('y_test',y_test)]:
    print(n, a.shape, a.dtype)

print(val_meta.head()); print(scaler)

Mounted at /content/drive
X_train (2770824, 24, 7) float32
X_val (741034, 24, 7) float32
X_test (735451, 24, 7) float32
y_train (2770824,) float32
y_val (741034,) float32
y_test (735451,) float32
      place             datetime
0  Air Itam  2023-04-13 16:00:00
1  Air Itam  2023-04-13 17:00:00
2  Air Itam  2023-04-13 18:00:00
3  Air Itam  2023-04-13 19:00:00
4  Air Itam  2023-04-13 20:00:00
                 place  solar_mean   solar_std
0             Air Itam  210.325617  286.021714
1          Ampang Jaya  197.509221  265.544312
2           Ayer Keroh  200.795970  270.202260
3   Bandar Puteri Jaya  214.346427  288.503132
4   Bandar Putra Kulai  198.301502  267.783049
..                 ...         ...         ...
62           Tamparuli  223.363465  300.622362
63         Tanjung Aru  207.048226  274.919525
64      Tanjung Bungah  211.956378  285.842560
65               Tawau  205.002165  270.527211
66             Tegupeh  216.625046  291.358189

[67 rows x 3 columns]


In [ ]:
# 1. Is y scaled? (z-scored would be mean≈0, std≈1)
print('y_train mean/std/min/max:', y_train.mean(), y_train.std(), y_train.min(), y_train.max())

# 2. Any NaNs?
print('NaN in X:', np.isnan(X_train[:200000]).any(), '| NaN in y:', np.isnan(y_train).any())

# 3. Meta lines up with y?
print(len(val_meta)==len(y_val), len(test_meta)==len(y_test))

# 4. Every station in meta has scaler stats?
stats = scaler.set_index('place')
print('Missing stations:', set(val_meta.place) - set(stats.index), set(test_meta.place) - set(stats.index))
print('Stations in val/test:', val_meta.place.nunique(), test_meta.place.nunique())

# 5. Helper: scaled -> real W/m²
def to_wm2(y_scaled, meta):
    mu = meta['place'].map(stats['solar_mean']).values
    sd = meta['place'].map(stats['solar_std']).values
    return y_scaled * sd + mu

print(to_wm2(y_val[:5], val_meta.iloc[:5]))   # should look like realistic GHI values

y_train mean/std/min/max: -0.0029063546 0.9985686 -0.7741195 3.1397836
NaN in X: False | NaN in y: False
True True
Missing stations: set() set()
Stations in val/test: 67 67
[ 2.48679993e+02  8.47200068e+01  3.69999926e+00 -1.06054526e-05
 -1.06054526e-05]


In [ ]:
def metrics_table(p, t, meta, day_thr=50):
    df = pd.DataFrame({'place': meta['place'].values, 'p': p, 't': t})
    def m(g):
        e = g.p - g.t
        rmse = np.sqrt((e**2).mean())
        day = g.t > day_thr
        return pd.Series({
            'RMSE': rmse,
            'MAE': e.abs().mean(),
            'R2': 1 - (e**2).sum() / ((g.t - g.t.mean())**2).sum(),
            'nRMSE': rmse / g.t.mean(),
            'MAPE_day%': (e[day].abs() / g.t[day]).mean() * 100,
        })
    per_station = df.groupby('place')[['p','t']].apply(m)
    return per_station, per_station.mean()

# --- Persistence baseline: next hour = last observed GHI ---
cors = [np.corrcoef(X_val[:200000, -1, j], y_val[:200000])[0, 1] for j in range(7)]
GHI_IDX = int(np.argmax(cors))
print(np.round(cors, 3), '-> GHI column:', GHI_IDX)

t_test = to_wm2(y_test, test_meta)
p_pers = np.clip(to_wm2(X_test[:, -1, GHI_IDX], test_meta), 0, None)
ps_pers, overall_pers = metrics_table(p_pers, t_test, test_meta)
print(overall_pers)

[ 0.371  0.176  0.123 -0.348  0.226  0.371  0.937] -> GHI column: 6
RMSE         96.326637
MAE          61.373107
R2            0.876357
nRMSE         0.469623
MAPE_day%    46.658556
dtype: float64


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print(tf.config.list_physical_devices('GPU'))   # should show a GPU
tf.keras.utils.set_random_seed(42)

BATCH = 1024
def make_ds(X, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle: ds = ds.shuffle(100_000, seed=42)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(X_train, y_train, shuffle=True)
val_ds   = make_ds(X_val, y_val)

def build_lstm():
    m = models.Sequential([
        layers.Input(shape=(24, 7)),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(32),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu'),
        layers.Dense(1),
    ], name='lstm')
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return m

lstm = build_lstm()
lstm.summary()

cbs = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=2, monitor='val_loss'),
    callbacks.ModelCheckpoint(P+'lstm_24h_best.keras', save_best_only=True, monitor='val_loss'),
]

hist = lstm.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=cbs)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Model: "lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 24, 64)         │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,393 (122.63 KB)

 Trainable params: 31,393 (122.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0337 - mae: 0.1068 - val_loss: 0.0181 - val_mae: 0.0776 - learning_rate: 0.0010
Epoch 2/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 42s 16ms/step - loss: 0.0167 - mae: 0.0717 - val_loss: 0.0207 - val_mae: 0.0857 - learning_rate: 0.0010
Epoch 3/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 81s 15ms/step - loss: 0.0153 - mae: 0.0661 - val_loss: 0.0210 - val_mae: 0.0961 - learning_rate: 0.0010
Epoch 4/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 43s 16ms/step - loss: 0.0146 - mae: 0.0630 - val_loss: 0.0244 - val_mae: 0.1000 - learning_rate: 5.0000e-04
Epoch 5/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 45s 17ms/step - loss: 0.0144 - mae: 0.0621 - val_loss: 0.0254 - val_mae: 0.1016 - learning_rate: 5.0000e-04
Epoch 6/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 44s 16ms/step - loss: 0.0141 - mae: 0.0610 - val_loss: 0.0287 - val_mae: 0.1041 - learning_rate: 2.5000e-04


In [ ]:
def evaluate(model, X, y, meta):
    pred = model.predict(X, batch_size=4096).ravel()
    p = np.clip(to_wm2(pred, meta), 0, None)
    t = to_wm2(y, meta)
    rmse = np.sqrt(np.mean((p-t)**2))
    mae  = np.mean(np.abs(p-t))
    r2   = 1 - np.sum((p-t)**2)/np.sum((t-t.mean())**2)
    nrmse = rmse / t.mean()
    return dict(RMSE=rmse, MAE=mae, R2=r2, nRMSE=nrmse), p, t

res_lstm, p_lstm, t_test = evaluate(lstm, X_test, y_test, test_meta)
print(res_lstm)

180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step
{'RMSE': np.float64(36.91919101214001), 'MAE': np.float64(21.492255289182204), 'R2': np.float64(0.9823911136041196), 'nRMSE': np.float64(0.17884723985066103)}


In [ ]:
j = GHI_IDX
# Leak if ~0: the window's last step already contains the target
print('own y vs last step:', np.median(np.abs(X_val[:, -1, j] - y_val)))
# Should be ~0: the next sample's last step equals this sample's target
print('next sample vs y  :', np.median(np.abs(X_val[1:, -1, j] - y_val[:-1])))

own y vs last step: 0.06529653
next sample vs y  : 0.0


In [ ]:
ps_lstm, overall_lstm = metrics_table(p_lstm, t_test, test_meta)
print(pd.concat([overall_lstm.rename('LSTM'), overall_pers.rename('Persistence')], axis=1))

                LSTM  Persistence
RMSE       36.145805    96.326637
MAE        21.153903    61.373107
R2          0.982454     0.876357
nRMSE       0.176571     0.469623
MAPE_day%  12.090046    46.658556


In [ ]:
# 1. Shuffle once across all samples
perm = np.random.RandomState(42).permutation(len(X_train))
X_train, y_train = X_train[perm], y_train[perm]

# 2. Rebuild datasets (no more small shuffle buffer)
train_ds = make_ds(X_train, y_train, shuffle=False)
val_ds   = make_ds(X_val, y_val)

# 3. Model matching Table 3-2
def build_lstm():
    m = models.Sequential([
        layers.Input(shape=(24, 7)),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(32),
        layers.Dropout(0.2),
        layers.Dense(1),
    ], name='lstm_v2')
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return m

tf.keras.utils.set_random_seed(42)
lstm = build_lstm()
cbs = [
    callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss'),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=2, monitor='val_loss'),
    callbacks.ModelCheckpoint(P+'lstm_24h_v2_best.keras', save_best_only=True, monitor='val_loss'),
]
hist = lstm.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=cbs)

Epoch 1/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 41s 14ms/step - loss: 0.0387 - mae: 0.1232 - val_loss: 0.0139 - val_mae: 0.0635 - learning_rate: 0.0010
Epoch 2/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 36s 13ms/step - loss: 0.0233 - mae: 0.1001 - val_loss: 0.0132 - val_mae: 0.0595 - learning_rate: 0.0010
Epoch 3/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 41s 14ms/step - loss: 0.0226 - mae: 0.0979 - val_loss: 0.0133 - val_mae: 0.0592 - learning_rate: 0.0010
Epoch 4/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 37s 13ms/step - loss: 0.0222 - mae: 0.0968 - val_loss: 0.0133 - val_mae: 0.0608 - learning_rate: 0.0010
Epoch 5/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 37s 14ms/step - loss: 0.0218 - mae: 0.0956 - val_loss: 0.0128 - val_mae: 0.0581 - learning_rate: 5.0000e-04
Epoch 6/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 41s 14ms/step - loss: 0.0216 - mae: 0.0953 - val_loss: 0.0127 - val_mae: 0.0571 - learning_rate: 5.0000e-04
Epoch 7/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 37s 14ms/step - loss: 0.0216 - mae: 0.0950 - val_loss: 0.0125 - val_mae: 0

In [ ]:
def evaluate(model, X, y, meta):
    pred = model.predict(X, batch_size=4096).ravel()
    p = np.clip(to_wm2(pred, meta), 0, None)
    t = to_wm2(y, meta)
    rmse = np.sqrt(np.mean((p-t)**2))
    mae  = np.mean(np.abs(p-t))
    r2   = 1 - np.sum((p-t)**2)/np.sum((t-t.mean())**2)
    nrmse = rmse / t.mean()
    return dict(RMSE=rmse, MAE=mae, R2=r2, nRMSE=nrmse), p, t

res_lstm, p_lstm, t_test = evaluate(lstm, X_test, y_test, test_meta)
print(res_lstm)

180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step
{'RMSE': np.float64(30.34956470565395), 'MAE': np.float64(14.754745002761357), 'R2': np.float64(0.9881003969983382), 'nRMSE': np.float64(0.14702206980890775)}


In [ ]:
ps_lstm, overall_lstm = metrics_table(p_lstm, t_test, test_meta)
print(pd.concat([overall_lstm.rename('LSTM'), overall_pers.rename('Persistence')], axis=1))

                LSTM  Persistence
RMSE       29.616852    96.326637
MAE        14.639258    61.373107
R2          0.988156     0.876357
nRMSE       0.144823     0.469623
MAPE_day%   8.733712    46.658556


In [ ]:
res = pd.concat([overall_lstm.rename('LSTM'), overall_pers.rename('Persistence')], axis=1)
res.to_csv(P+'results_overall_so_far.csv')
ps_lstm.to_csv(P+'per_station_lstm_v2.csv')
ps_pers.to_csv(P+'per_station_persistence.csv')

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os, numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras import layers, models, callbacks
P = '/content/drive/MyDrive/fyp_solar/'
print(sorted(os.listdir(P)))          # <- what's saved in your folder

# --- data ---
X_train = np.load(P+'train_X_24h_v2.npy'); y_train = np.load(P+'train_y_24h_v2.npy')
X_val   = np.load(P+'val_X_24h_v2.npy');   y_val   = np.load(P+'val_y_24h_v2.npy')
X_test  = np.load(P+'test_X_24h_v2.npy');  y_test  = np.load(P+'test_y_24h_v2.npy')
val_meta  = pd.read_csv(P+'val_meta_24h.xls')
test_meta = pd.read_csv(P+'test_meta_24h.xls')
stats = pd.read_csv(P+'solar_scaler_stats.xls').set_index('place')

# --- helpers ---
def to_wm2(y_scaled, meta):
    mu = meta['place'].map(stats['solar_mean']).values
    sd = meta['place'].map(stats['solar_std']).values
    return y_scaled * sd + mu

def metrics_table(p, t, meta, day_thr=50):
    df = pd.DataFrame({'place': meta['place'].values, 'p': p, 't': t})
    def m(g):
        e = g.p - g.t
        rmse = np.sqrt((e**2).mean())
        day = g.t > day_thr
        return pd.Series({
            'RMSE': rmse,
            'MAE': e.abs().mean(),
            'R2': 1 - (e**2).sum() / ((g.t - g.t.mean())**2).sum(),
            'nRMSE': rmse / g.t.mean(),
            'MAPE_day%': (e[day].abs() / g.t[day]).mean() * 100,
        })
    per_station = df.groupby('place')[['p','t']].apply(m)
    return per_station, per_station.mean()

# --- shuffle once + datasets ---
perm = np.random.RandomState(42).permutation(len(X_train))
X_train, y_train = X_train[perm], y_train[perm]
BATCH = 1024
def make_ds(X, y):
    return tf.data.Dataset.from_tensor_slices((X, y)).batch(BATCH).prefetch(tf.data.AUTOTUNE)
train_ds, val_ds = make_ds(X_train, y_train), make_ds(X_val, y_val)

# --- reload LSTM + persistence, rebuild the comparison ---
GHI_IDX = 6
t_test = to_wm2(y_test, test_meta)

lstm = models.load_model(P+'lstm_24h_v2_best.keras')
p_lstm = np.clip(to_wm2(lstm.predict(X_test, batch_size=4096).ravel(), test_meta), 0, None)
ps_lstm, overall_lstm = metrics_table(p_lstm, t_test, test_meta)

p_pers = np.clip(to_wm2(X_test[:, -1, GHI_IDX], test_meta), 0, None)
ps_pers, overall_pers = metrics_table(p_pers, t_test, test_meta)

print(pd.concat([overall_lstm.rename('LSTM'), overall_pers.rename('Persistence')], axis=1))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['lstm_24h_best.keras', 'lstm_24h_v2_best.keras', 'per_station_lstm_v2.csv', 'per_station_persistence.csv', 'results_overall_so_far.csv', 'solar_scaler_stats.xls', 'test_X_24h_v2.npy', 'test_meta_24h.xls', 'test_y_24h_v2.npy', 'train_X_24h_v2.npy', 'train_y_24h_v2.npy', 'val_X_24h_v2.npy', 'val_meta_24h.xls', 'val_y_24h_v2.npy']
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step
                LSTM  Persistence
RMSE       29.616852    96.326637
MAE        14.639258    61.373107
R2          0.988156     0.876357
nRMSE       0.144823     0.469623
MAPE_day%   8.733712    46.658556


In [4]:
def build_cnn_bilstm():
    m = models.Sequential([
        layers.Input(shape=(24, 7)),
        layers.Conv1D(64, kernel_size=3, padding='same', activation='relu'),
        layers.MaxPooling1D(pool_size=2),                # 24 -> 12 steps
        layers.Bidirectional(layers.LSTM(64)),
        layers.Dropout(0.2),
        layers.Dense(1),
    ], name='cnn_bilstm')
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return m

tf.keras.utils.set_random_seed(42)
cnn_bilstm = build_cnn_bilstm()
cnn_bilstm.summary()

cbs = [
    callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss'),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-5, monitor='val_loss'),
    callbacks.ModelCheckpoint(P+'cnn_bilstm_24h_best.keras', save_best_only=True, monitor='val_loss'),
]
hist2 = cnn_bilstm.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=cbs)

Model: "cnn_bilstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 24, 64)         │         1,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 12, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 67,585 (264.00 KB)

 Trainable params: 67,585 (264.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 41s 13ms/step - loss: 0.0262 - mae: 0.0985 - val_loss: 0.0139 - val_mae: 0.0662 - learning_rate: 0.0010
Epoch 2/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 34s 13ms/step - loss: 0.0169 - mae: 0.0782 - val_loss: 0.0133 - val_mae: 0.0616 - learning_rate: 0.0010
Epoch 3/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 33s 12ms/step - loss: 0.0162 - mae: 0.0755 - val_loss: 0.0130 - val_mae: 0.0587 - learning_rate: 0.0010
Epoch 4/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 33s 12ms/step - loss: 0.0158 - mae: 0.0741 - val_loss: 0.0130 - val_mae: 0.0606 - learning_rate: 0.0010
Epoch 5/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 41s 13ms/step - loss: 0.0156 - mae: 0.0732 - val_loss: 0.0126 - val_mae: 0.0558 - learning_rate: 0.0010
Epoch 6/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 34s 13ms/step - loss: 0.0155 - mae: 0.0726 - val_loss: 0.0129 - val_mae: 0.0564 - learning_rate: 0.0010
Epoch 7/30
2706/2706 ━━━━━━━━━━━━━━━━━━━━ 33s 12ms/step - loss: 0.0154 - mae: 0.0722 - val_loss: 0.0127 - val_mae: 0.0564 - 

In [5]:
pred = cnn_bilstm.predict(X_test, batch_size=4096).ravel()
p_cnn = np.clip(to_wm2(pred, test_meta), 0, None)
ps_cnn, overall_cnn = metrics_table(p_cnn, t_test, test_meta)

res = pd.concat([overall_lstm.rename('LSTM'),
                 overall_cnn.rename('CNN-BiLSTM'),
                 overall_pers.rename('Persistence')], axis=1)
print(res)

# save so a disconnect can't lose it
res.to_csv(P+'results_overall_v2.csv')
ps_cnn.to_csv(P+'per_station_cnn_bilstm.csv')

180/180 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step
                LSTM  CNN-BiLSTM  Persistence
RMSE       29.616852   28.945891    96.326637
MAE        14.639258   14.104123    61.373107
R2          0.988156    0.988691     0.876357
nRMSE       0.144823    0.141529     0.469623
MAPE_day%   8.733712    8.381789    46.658556


In [6]:
cmp = pd.DataFrame({
    'LSTM': ps_lstm['RMSE'],
    'CNN-BiLSTM': ps_cnn['RMSE'],
    'Persistence': ps_pers['RMSE'],
})
cmp['CNN wins'] = cmp['CNN-BiLSTM'] < cmp['LSTM']

print('CNN-BiLSTM beats LSTM at', cmp['CNN wins'].sum(), 'of', len(cmp), 'stations')
print('\nBest 5 stations:');  print(cmp.sort_values('CNN-BiLSTM').head(5).round(1))
print('\nWorst 5 stations:'); print(cmp.sort_values('CNN-BiLSTM').tail(5).round(1))
print('\nnRMSE spread across stations:'); print(ps_cnn['nRMSE'].describe().round(3))

cmp.to_csv(P+'per_station_rmse_compare.csv')

CNN-BiLSTM beats LSTM at 66 of 67 stations

Best 5 stations:
              LSTM  CNN-BiLSTM  Persistence  CNN wins
place                                                
Muallim       24.3        23.6         97.0      True
Sejingkat     27.0        25.4         92.2      True
Nilai Impian  25.8        25.4        100.8      True
Gita Bayu     26.4        25.5         94.9      True
Sungai Ruan   26.6        25.9         91.6      True

Worst 5 stations:
                    LSTM  CNN-BiLSTM  Persistence  CNN wins
place                                                      
Tanjung Bungah      32.7        32.0         97.1      True
Johor Bahru         32.4        32.3         94.3      True
Sandakan            33.3        33.1         99.8      True
Taman Kulai Besar   34.3        33.5         94.8      True
Bandar Putra Kulai  34.4        33.9         93.7      True

nRMSE spread across stations:
count    67.000
mean      0.142
std       0.014
min       0.110
25%       0.132
50%       0